# TC_11: ResNet clean baseline
Full diagnostic pipeline on clean image data using ResNet18 model.
UQ: Deep Ensemble, MC Dropout
XAI: Grad-CAM

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch


sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_image_data
from src.data_diagnostics.quality_checks import check_image_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.image_models import train_resnet, evaluate_image_model
from src.inference_diagnostics.uncertainty import mc_dropout_prediction_img, deep_ensemble_prediction_img
from src.inference_diagnostics.explainability import gradcam_img, collect_gradcam_feedback
from src.inference_diagnostics.flagging import assign_flags, evaluate_flags
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report
from src.utils.visualization import plot_uncertainty_distribution, plot_flag_distribution, plot_gradcam_sample
from src.utils.config_loader import load_config, get_image_config, save_threshold
from src.utils.per_sample_logger import save_per_sample, build_experiment_id
from src.inference_diagnostics.explainability import collect_gradcam_feedback_resumable

config = load_config()
tracker = PerformanceTracker()

dataset_name = get_image_config(config)['name']
dataset_short = get_image_config(config)['short_name']
model_name = 'resnet18'
test_case = 'TC11'
review_count = config['stage3_inference']['explainability']['gradcam']['max_samples_to_explain']
experiment_id = build_experiment_id(dataset_short, model_name, test_case)
print(f"Experiment: {experiment_id}, reviewing {review_count} samples")

Experiment: cifar10_resnet18_TC11, reviewing 100 samples


### Load image data and EDA


In [2]:
train_set, test_set = load_image_data(config)
report = check_image_quality(train_set, config)
plot_class_distribution(report, f"ResNet {dataset_name} Baseline", config)

Loading dataset.
Dataset: CIFAR-10
Loaded: 50000 train and 10000 test
Image size: 32, Channels: 3
Classes: 10
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
EDA started for image data.
Total images: 50000
Classes: 10
Distribution: {0: 5000, 1: 5000, 2: 5000, 3: 5000, 4: 5000, 5: 5000, 6: 5000, 7: 5000, 8: 5000, 9: 5000}
Imbalance ratio: 1.0
EDA completed for CIFAR-10
Saved: results/figures\class_distribution_resnet_cifar-10_baseline.png


### Train CNN baseline

In [3]:
tracker.start_performance_track()
resnet_model = train_resnet(train_set, config)
tracker.stop_performance_track("ResNet training")

tracker.start_performance_track()
resnet_accuracy, resnet_prediction, resnet_report = evaluate_image_model(resnet_model, test_set, config)
tracker.stop_performance_track("ResNet evaluation")

Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
ResNet training: Time:704.17s, Memory:827.62MB
 Image model evaluation started.
{'0': {'precision': 0.7051282051282052, 'recall': 0.77, 'f1-score': 0.7361376673040153, 'support': 1000.0}, '1': {'precision': 0.8, 'recall': 0.796, 'f1-score': 0.7979949874686717, 'support': 1000.0}, '2': {'precision': 0.6357758620689655, 'recall': 0.59, 'f1-score': 0.6120331950207469, 'support': 1000.0}, '3': {'precision': 0.4903581267217631, 'recall': 0.534, 'f1-score': 0.511249401627573, 'support': 1000.0}, '4': {'precision': 0.6482694106641721, 'recall': 0.693, 'f1-score': 0.6698888351860802, 'support': 1000.0}, '5': {'precision': 0.6135593220338983, 'recall': 0.543, 'f1-score': 0.5761273209549072, 'support': 1000.0}, '6': {'precision': 0.7541789577187807, 'recall': 0.767, 'f1-score': 0.7605354486861676, 'support': 1000.0}, '7': {'precision': 0.7913043478260869, 'recall': 0.728, 'f1-score': 0.7583

{'operation': 'ResNet evaluation', 'time_seconds': 5.45, 'memory_usage': 12.3}

### Uncertainty Quantification (MC Dropout + Deep Ensemble)

In [4]:
# MC Dropout
tracker.start_performance_track()
mc_means_probabilities, mc_uncertainty = mc_dropout_prediction_img(resnet_model, test_set, config)
tracker.stop_performance_track("ResNet MC Dropout")

mc_threshold = round(float(np.percentile(mc_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'mc', mc_threshold)
print(f"MC Dropout uncertainty status:")
print(f"Mean: {mc_uncertainty.mean()}")
print(f"Max: {mc_uncertainty.max()}")
print(f" 90th percentile (threshold): {mc_threshold}")

plot_uncertainty_distribution(mc_uncertainty, "ResNet MC Dropout", mc_threshold, config)

MC Dropout started.
Pass 10/50 done.
Pass 20/50 done.
Pass 30/50 done.
Pass 40/50 done.
Pass 50/50 done.
MC Dropout finished for images.
ResNet MC Dropout: Time:207.53s, Memory:19.55MB
Threshold saved: cifar10_resnet18_mc = 0.0
MC Dropout uncertainty status:
Mean: 3.2668719285311454e-08
Max: 7.259173173679301e-08
 90th percentile (threshold): 0.0
Saved: results/figures\uncertainty_distribution_resnet_mc_dropout.png


In [5]:
# Deep Ensemble
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_img(train_resnet, config, test_set, train_set  )
tracker.stop_performance_track("ResNet Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
save_threshold(config, dataset_short, model_name, 'de', de_threshold)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "ResNet Deep Ensemble", de_threshold, config)

Deep Ensemble started for images.
Training model with seed 0
Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
Training model with seed 1
Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
Training model with seed 2
Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
Training model with seed 3
Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
Training model with seed 4
Training resnet model started.
Epoch 5/20
Epoch 10/20
Epoch 15/20
Epoch 20/20
ResNet18 training completed.
Deep Ensemble finished for images.
ResNet Deep Ensemble prediction: Time:3714.66s, Memory:32.25MB
Threshold saved: cifar10_resnet18_de = 0.1206
Deep Ensembl uncertainty status:
Mean: 0.056228380650281906
Max: 0.18721210956573486
 90th percentile (threshold): 0.1206
Saved: results/figures\uncerta

### Explainability (Grad-CAM and Human feedback)

In [6]:
tracker.start_performance_track()
heatmaps = gradcam_img(resnet_model, test_set, config)
tracker.stop_performance_track("ResNet Grad-CAM")

# Show grid for expert review
plot_gradcam_sample(test_set, heatmaps, review_count, config, f"ResNet {dataset_name} Baseline", save=True)


GradCam started.
*****************************************************************************************
GradCam finished. Explained: 50 / 100 samples.
GradCam finished. Explained: 100 / 100 samples.
GradCam finished. Explained: 100 samples.
ResNet Grad-CAM: Time:1.17s, Memory:41.41MB
Saved: results/figures\gradcam_sample_resnet_cifar-10_baseline.png


### Collect human feedback

In [7]:
# consistency_scores, correct, incorrect, partial = collect_gradcam_feedback(20)
# print(f"Human feedback: correct: {correct}, incorrect: {incorrect}, partial: {partial}")
consistency_scores, correct, incorrect, partial = collect_gradcam_feedback_resumable(
    review_count, config, experiment_id, batch_size=10
)

Scoring: 1 = Correct, 2 = Partial, 3 = Incorrect
Review in batches of 10. Type 'stop' to pause and resume later.

  Saved. Progress: 10/100
  Saved. Progress: 20/100
  Saved. Progress: 30/100
  Saved. Progress: 40/100
  Saved. Progress: 50/100
  Saved. Progress: 60/100
  Saved. Progress: 70/100
  Saved. Progress: 80/100
  Saved. Progress: 90/100
  Saved. Progress: 100/100
Correct: 12, Incorrect: 35, Partial: 53


### Flagging (UQ + Grad-CAM feedback)

In [8]:
# Get true  labels for samples
y_true_reviewed = np.array([test_set[i][1] for i in range(review_count)])

# MC Dropout
mc_y_pred_reviewed = mc_means_probabilities[:review_count].argmax(axis=1)
mc_flags_reviewed = assign_flags(mc_uncertainty[:review_count], consistency_scores, mc_threshold, 0.5)
mc_flag_result_reviewed = evaluate_flags(mc_flags_reviewed, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_reviewed, "ResNet MC + Grad-CAM Reviewed", config)

# Deep Ensemble
de_y_pred_reviewed = de_means_probabilities[:review_count].argmax(axis=1)
de_flags_reviewed = assign_flags(de_uncertainty[:review_count], consistency_scores, de_threshold, 0.5)
de_flag_result_reviewed = evaluate_flags(de_flags_reviewed, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_reviewed, "ResNet DE + Grad-CAM Reviewed", config)

RED: Count: 35 Accuracy:11.4%
YELLOW: Count: 65 Accuracy:21.5%
GREEN: Count: 0 Accuracy:0.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_resnet_mc_+_grad-cam_reviewed.png
RED: Count: 7 Accuracy:57.1%
YELLOW: Count: 35 Accuracy:74.3%
GREEN: Count: 58 Accuracy:84.5%
Flagging system is working
Saved: results/figures\flagging_distribution_resnet_de_+_grad-cam_reviewed.png


### Full flagging with constant consistency for all sample

In [9]:
# Full test set flagging (UQ only, no human review)
fully_y_true = np.array([test_set[i][1] for i in range(len(test_set))])
fake_consistency = np.ones(len(mc_uncertainty))

# MC only
full_mc_y_prediction = mc_means_probabilities.argmax(axis = 1)
mc_flags_full = assign_flags(mc_uncertainty, fake_consistency, mc_threshold, 0.5)
mc_full_result = evaluate_flags(mc_flags_full, full_mc_y_prediction, fully_y_true)
plot_flag_distribution(mc_flags_full, "ResNet MC UQ Only Full", config)

# Deep Ensemble only
full_de_y_prediction = de_means_probabilities.argmax(axis = 1)
de_flags_full = assign_flags(de_uncertainty, fake_consistency, de_threshold, 0.5)
de_full_result = evaluate_flags(de_flags_full, full_de_y_prediction, fully_y_true)
plot_flag_distribution(de_flags_full, "ResNet DE UQ Only Full", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 10000 Accuracy:14.4%
GREEN: Count: 0 Accuracy:0.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_resnet_mc_uq_only_full.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 1003 Accuracy:45.9%
GREEN: Count: 8997 Accuracy:82.4%
Flagging system is working
Saved: results/figures\flagging_distribution_resnet_de_uq_only_full.png


In [10]:
# 20 sample test set flagging (UQ only, no human review)
fake_consistency_reviewed = np.ones(review_count)

# MC only
mc_flags_uq_only = assign_flags(mc_uncertainty[:review_count], fake_consistency_reviewed, mc_threshold, 0.5)
mc_20_result = evaluate_flags(mc_flags_uq_only, mc_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(mc_flags_uq_only, "ResNet MC UQ Only reviewed", config)

# Deep Ensemble only
de_flags_uq_only = assign_flags(de_uncertainty[:review_count], fake_consistency_reviewed, de_threshold, 0.5)
de_20_result = evaluate_flags(de_flags_uq_only, de_y_pred_reviewed, y_true_reviewed)
plot_flag_distribution(de_flags_uq_only, "ResNet DE UQ Only reviewed", config)

RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 100 Accuracy:18.0%
GREEN: Count: 0 Accuracy:0.0%
Flagging system is not working, needs threshold adjustment
Saved: results/figures\flagging_distribution_resnet_mc_uq_only_reviewed.png
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 14 Accuracy:50.0%
GREEN: Count: 86 Accuracy:83.7%
Flagging system is working
Saved: results/figures\flagging_distribution_resnet_de_uq_only_reviewed.png


In [11]:
save_per_sample(
    config,
    experiment_id,
    y_true=y_true_reviewed,
    y_pred=mc_y_pred_reviewed,
    mc_uncertainty=mc_uncertainty[:review_count],
    de_uncertainty=de_uncertainty[:review_count],
    consistency=consistency_scores,
)

Per sample results saved: results/per_sample\cifar10_resnet18_TC11.csv (100 rows)


,sample_id,y_true,y_pred,correct,mc_uncertainty,de_uncertainty,consistency
0,0,0,0,1,2.125745e-08,0.000172,0.5
1,1,0,5,0,2.980471e-08,0.078020,0.5
2,2,0,2,0,5.423893e-08,0.074482,0.0
3,3,0,6,0,2.386188e-08,0.104503,0.5
4,4,0,0,1,3.018404e-08,0.146342,0.0
...,...,...,...,...,...,...,...
95,95,0,2,0,3.576437e-08,0.005787,0.5
96,96,0,8,0,4.172472e-08,0.012163,0.5
97,97,0,0,1,3.173827e-08,0.002065,0.5
98,98,0,0,1,6.556569e-08,0.000077,0.0


### Save golden baseline report

In [12]:
resnet_baseline = {
    'model': 'ResNet18',
    'accuracy': round(resnet_accuracy, 4),
    'classification_report': resnet_report,
    'mc_uncertainty': {
        'mean': round(float(mc_uncertainty.mean()), 8),
        'max': round(float(mc_uncertainty.max()), 8),
        'threshold': mc_threshold
    },
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold
    },
    'gradcam_feedback':{
        'correct': int(correct),
        'incorrect': int(incorrect),
        'partial': int(partial)
    },
    'flagging_mc': mc_flag_result_reviewed,
    'flagging_de': de_flag_result_reviewed,
    'flagging_mc_20_mc_only': mc_20_result,
    'flagging_mc_20_de_only': de_20_result,
    'flagging_mc_full': mc_full_result,
    'flagging_de_full': de_full_result,
    'mc_vs_mc_plus_xai': round(mc_flag_result_reviewed['GREEN']['accuracy'] - mc_20_result['GREEN']['accuracy'], 4),
    'de_vs_de_plus_xai': round(de_flag_result_reviewed['GREEN']['accuracy'] - de_20_result['GREEN']['accuracy'], 4),
    'performance': tracker.get_results()
}

report_output = generate_report(config, f"{get_image_config(config)['name']} - ResNet golden baseline", stage2_result=resnet_baseline)
save_report(report_output, f'{dataset_short.upper()}_TC_11_ResNet_Golden_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
